# 3단계 — Dropout 비율 (이 단계는 Dropout 비율 하나만 바꾼다)

학습률 단계가 끝났다 (Adam 기준선 1e-3 유지). 다음은 정규화다. 한 번에 하나만 바꾸는 원칙대로 **이번 단계는 Dropout 비율만 바꾼다.** 에폭 수·조기 종료·데이터 증강은 이 단계가 끝난 뒤 하나씩 따로 본다. 지금까지 train 정확도는 계속 오르는데 val은 2~3에폭 뒤부터 정체했다 (Adam 1e-3, 3시드 평균: Keras train 75.4% 대 val 39.3%, PyTorch 67.2% 대 40.8%). 과적합을 줄이는 가장 간단한 손잡이부터 만져 본다.

| | 값 |
|---|---|
| 프레임워크 | Keras, PyTorch |
| 바꾸는 것 | Dropout 비율 0.0 · 0.25 · **0.5(현재 기준선)** · 0.75 · 0.9 (0.0은 Dropout을 끈 것) |
| 시드 | 0 한 번씩 훑어본다. 시드를 더하려면 셀 1의 `SEEDS`를 `(0, 1, 2)`로 바꾼다 |
| 실행 횟수 | 5 × 2 = **10번** |
| 그 밖의 조건 | 2단계와 같다: He 초기화, 2 Conv(32·64) + Dropout + Dense(11), Adam lr 1e-3, batch 32, **5 epoch 고정**, 증강 없음, 학습률 고정(스케줄러 없음) |
| 소요 시간 | 대략 35분 (Keras 5번 ≈ 13분, PyTorch 5번 ≈ 23분). 시드 3개로 늘리면 약 1시간 45분 |

**Keras·PyTorch 셀은 2단계에서 실제로 돌아간 코드 그대로이고, Dropout 비율을 넣는 줄만 바뀌었다** (`Dropout(0.5)` → `Dropout(dropout)`과 그 값을 넘기는 줄). 0.5 행은 2단계 Adam 1e-3과 같은 설정이라 같은 값이 나와야 한다.

## Dropout이란

**학습할 때마다 일부 값을 무작위로 0으로 만들어서, 모델이 특정 값에 기대는 걸 막는 정규화 방법**이다. 이 모델에서는 Flatten 뒤, 출력층(Dense) 바로 앞의 57,600개 값 중 비율 p만큼을 매 배치마다 0으로 만든다. 평가할 때는 끄고 값의 크기만 맞춘다. 파라미터가 없어서 모델 크기는 그대로다 (653,003개).

기준선에 이미 0.5가 들어 있다. 그래서 이번 단계는 Dropout을 새로 "넣는" 게 아니라 **비율을 바꾸는** 단계이고, 0.0이 "Dropout 없음"이다. 위치도 그대로다 (출력층 앞 한 곳).

## 예상되는 작용

- 비율을 올리면 학습이 느려진다. 같은 5에폭 동안 train 정확도가 덜 오르고 train 대 val 격차가 줄어든다
- 비율이 0이면 과적합이 일찍, 크게 온다. val loss가 바닥을 찍는 에폭이 앞당겨지고 격차가 가장 크다
- 너무 크면(0.9) 신호가 너무 많이 가려져서 5에폭 안에 덜 배운 채 끝난다
- 다만 이 모델의 Dropout은 출력층 앞 한 곳뿐이다. 과적합의 격차(train 대 val 26~36%p)를 크게 줄이기에는 약할 수 있다

## 이 단계가 바꾸지 않는 것 (한계)

**5에폭 고정이다.** 에폭 수와 조기 종료는 Dropout과 별개의 변경이라서 이번에는 건드리지 않는다. 그 대가로, 비율을 올려 학습이 느려진 설정은 5에폭이 끝날 때까지 val loss가 바닥에 닿지 못할 수 있다. 그 경우 결과에는 "정규화 효과"와 "덜 학습된 효과"가 섞여 있다. 결과표 비고의 `최저 에폭=마지막`이 그런 설정이다 (학습률 단계의 Adam 1e-4도 그랬다). 에폭을 늘리거나 조기 종료를 거는 것은 이 단계가 끝난 뒤 따로 한 단계로 본다.

## 결론을 내는 규칙과 예상 (결과를 보기 전에 정해 둔다)

**시드 1개(지금 기본값)에서는 판정하지 않고 차이만 본다.** 2단계 훑어보기와 같다. 시드 0 한 번의 값은 높게 나오는 경향이 있었고(2단계 3시드 확인에서 Keras가 0.2~2.3%p 높았다), 같은 설정을 다시 돌려도 val이 1~2%p 움직인다. 비율을 바꿨을 때의 변화가 이보다 한참 크지 않으면 읽지 않는다.

**시드 3개를 모으면 다음 규칙으로 판정한다** (2단계 마무리와 같은 기준이다).

- 비율 하나하나를 기준선(0.5)과 같은 시드끼리 뺀다. 시드 전부에서 방향이 같고, 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 때만 "뚜렷함"이다
- 어떤 비율이 val acc와 val loss 최저 **둘 다, 두 프레임워크 모두(4칸 전부)**에서 기준선보다 뚜렷하게 낫다면 기준선을 그 비율로 옮긴다. 여러 개면 val acc 평균이 가장 높은 것이다. 아니면 0.5를 유지한다
- 4칸 전부에서 기준선이 뚜렷하게 낫다면 "기준선이 뚜렷하게 낫다"로 적는다 (0.0이면 Dropout 0.5가 끈 것보다 낫다는 뜻이다)
- train − val 격차와 test 정확도는 보고용이고 판정에 쓰지 않는다

**재현 확인 (0.5 행).** 0.5 행은 2단계 Adam 1e-3과 같은 설정이다. 시드 0이면 2단계 시드 0(Keras train 77.6 · val 41.5 · test 45.3, PyTorch 66.8 · 40.1 · 42.2), 시드 3개면 2단계 3시드 평균(Keras 75.4 · 39.3 · 43.0, PyTorch 67.2 · 40.8 · 43.4)과 재실행 흔들림(val 1~2%p) 안에서 맞아야 한다. 크게 벗어나면 코드나 환경이 달라진 것이므로 비율 비교를 읽기 전에 원인부터 확인한다.

### 예상 (가설 — 결과로 확인한다)

1. **비율이 클수록 5에폭 끝의 train 정확도가 낮고 train 대 val 격차가 작을 것이다.** 같은 5에폭 동안 학습이 느려지기 때문이다. 이 방향은 두 프레임워크 모두에서 보일 것으로 본다.
2. **Dropout을 끄면(0.0) 격차가 가장 크고, val loss 최저 에폭이 0.5와 같거나 앞설 것이다.** 기준선(0.5)에서 이미 2~3에폭이다.
3. **val 정확도의 차이는 작을 것이다.** 0.25 · 0.5 · 0.75 사이의 차이는 시드 범위(Keras 2.5~5.1%p, PyTorch 1.1~2.6%p) 안에 들 것으로 본다. 출력층 앞 한 곳의 Dropout만으로 일반화가 크게 바뀌지는 않을 것이다.
4. **비율을 올리면 val loss 최저는 내려가지만 val 정확도는 따라 오르지 않을 것이다.** 학습률 단계에서 Adam 1e-4가 그랬다 (val loss Keras 1.67 대 1.74, PyTorch 1.68 대 1.78인데 val 정확도는 구분되지 않았다). 과적합이 늦게 와서 바닥이 낮아지는 것과 정확도가 오르는 것은 별개라고 본다.
5. **0.9는 5에폭 안에 바닥에 닿지 못하고(`최저 에폭=마지막`) val 정확도가 낮게 나올 수 있다.**
6. 두 프레임워크에서 같은 방향이 나올 것이다.

## 실행 방법

위에서부터 셀을 차례로 돌린다 (셀 0 → 1 → 2 → 3 → 4). 자리를 비울 때는 셀 0에서 Drive 허용을 누른 뒤 "런타임 → 모두 실행"으로 이어서 돌려도 된다. 끝나면 셀 4가 요약을 낸다.

- 셀 1의 `[계획]`에 남은 실행 수와 시간 어림이 나온다. 이 단계의 결과는 Drive의 `food11_v2_dropout.csv`에 실행 하나가 끝날 때마다 한 줄씩 쌓이고, **런타임이 끊겨도 남는다.** 다시 돌리면 끝난 실행은 건너뛰고 이어서 한다. 처음부터 다시 하려면 그 파일을 지운다 (2단계의 `food11_v2_lr.csv`와는 별개 파일이다)
- 셀 2(Keras)와 셀 3(PyTorch)은 서로 독립이다. 셀 4(요약)는 있는 결과만으로 만든다
- 실행 하나마다 시작 줄 → 에폭 줄 5개 → 끝 줄이 나온다. 에폭 줄은 Keras 약 30초, PyTorch 약 55초 간격이다
- 시드를 더하려면 셀 1의 `SEEDS`를 `(0, 1, 2)`로 바꾸고 셀 1부터 다시 돌린다. 시드 0은 건너뛰고 1·2만 돈다. 시간을 줄이려면 `DROPOUTS`에서 시드 0에서 차이가 안 보인 비율을 뺀다 (0.5는 비교 기준이라 남겨 둔다)
- **셀 4의 출력을 통째로 붙여 주면 된다.**

## 어떻게 읽을까

셀 4는 출력 3개와 그래프 2개를 만든다: `[결과표]`(프레임워크마다) · `[기준선 대비]` · 판정 줄 · 그래프 `dropout_seeds.png`(비율에 따른 결과)와 `dropout_curves.png`(에폭별 곡선).

- **결과표**는 5에폭 끝 값이고 시드가 여러 개면 평균 [최소~최대]다. val loss 최저와 최저 에폭은 5에폭 중 val loss가 가장 낮았던 값이다. train − val은 5에폭 끝의 격차다 (작을수록 과적합이 약하다)
- 비고의 `최저 에폭=마지막`은 5에폭이 끝날 때도 val loss가 내려가는 중이었다는 뜻이다. 그 비율은 5에폭 기준으로 불리하게 평가됐을 수 있다
- 결과표 밑의 `[참고]` 줄은 2단계 Adam 1e-3(Dropout 0.5)의 값이다. 0.5 행이 그 값과 재실행 흔들림 안에서 맞는지 먼저 본다
- **기준선 대비**는 시드 1개이면 차이만 나오고 판정은 "판정 안 함"이다. 시드 3개가 되면 위 규칙대로 판정 줄이 나온다
- 그래프에서 점 하나가 시드 하나이고 같은 시드는 같은 가로 위치에 있다 (왼쪽부터 시드 0 · 1 · 2). 짧은 가로선이 평균이다. 곡선 그래프는 비율마다 한 줄이고 연한 색이 작은 비율이다

## 셀 0 — 데이터 복사 (최초 1회, 런타임 끊기면 다시)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Drive에서 직접 읽으면 학습이 수십 배 느려져서 로컬로 복사한다
if not os.path.isdir("/content/food11"):
    !cp -r /content/drive/MyDrive/food11 /content/food11

## 셀 1 — 공통 설정, 기록 함수, 실행 계획

In [ ]:
import csv
import gc
import json
import os
import time

import numpy as np

DATA_DIR = "/content/food11"
IMG, BATCH, EPOCHS = 128, 32, 5
NUM_CLASSES = 11
N_PARAMS = 653_003           # Dropout은 파라미터가 없어서 두 프레임워크 모두 이 값이어야 같은 모델이다

# 이 단계에서 바꾸는 것은 Dropout 비율 하나다. 나머지(He 초기화, 구조, Adam 1e-3, batch 32,
# 5 epoch 고정, 증강 없음)는 2단계 그대로다. 에폭 수와 조기 종료는 이 단계에서 건드리지 않는다.
OPTIMIZER, LR = "adam", 1e-3
DROPOUTS = (0.0, 0.25, 0.5, 0.75, 0.9)      # 0.5 = 현재 기준선, 0.0 = Dropout 없음
BASE_DROPOUT = 0.5
SEEDS = (0,)                 # (0, 1, 2)로 바꾸면 시드 0은 건너뛰고 1·2만 추가로 돈다
MIN_PER_RUN = {"keras": 2.5, "torch": 4.5}      # 실행 하나에 걸리는 시간 (분, 어림값)

# 실행 하나가 끝날 때마다 한 줄씩 쌓는다. 런타임이 끊겨도 남고,
# 다시 돌리면 끝난 실행은 건너뛴다. 처음부터 다시 하려면 이 파일을 지운다.
RESULTS = "/content/drive/MyDrive/food11_v2_dropout.csv"
FIELDS = ["framework", "optimizer", "lr", "dropout", "seed", "n_params", "minutes",
          "train_acc", "train_loss", "val_acc", "val_loss",    # 에폭별 값 (JSON 리스트)
          "test_acc", "test_loss"]


def done_runs():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS, newline="", encoding="utf-8") as f:
        return {(r["framework"], float(r["dropout"]), int(r["seed"]))
                for r in csv.DictReader(f)}


def save_row(row):
    is_new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)


def make_row(framework, dropout, seed, n_params, minutes, hist, test_loss, test_acc):
    return dict(framework=framework, optimizer=OPTIMIZER, lr=LR, dropout=dropout, seed=seed,
                n_params=n_params, minutes=round(minutes, 2),
                train_acc=json.dumps(hist["train_acc"]),
                train_loss=json.dumps(hist["train_loss"]),
                val_acc=json.dumps(hist["val_acc"]),
                val_loss=json.dumps(hist["val_loss"]),
                test_acc=test_acc, test_loss=test_loss)


def tag(framework, dropout, seed):
    return f"[{framework} · dropout {dropout} · 시드 {seed}]"


def report(row):
    tr = json.loads(row["train_acc"])[-1]
    va = json.loads(row["val_acc"])[-1]
    diverged = not np.isfinite(json.loads(row["val_loss"])[-1])
    print(f"{tag(row['framework'], float(row['dropout']), row['seed'])} "
          f"train {tr:.3f} · val {va:.3f} · test {float(row['test_acc']):.3f} "
          f"· {row['minutes']:.1f}분" + (" · 발산(loss NaN)" if diverged else ""))


def plan():
    done = done_runs()
    print(f"[계획] Dropout {', '.join(str(d) for d in DROPOUTS)} × 시드 {', '.join(map(str, SEEDS))}"
          " — 5에폭 고정, CSV에 이미 있는 실행은 건너뛴다")
    for fw, name in (("keras", "Keras"), ("torch", "PyTorch")):
        todo = [1 for s in SEEDS for d in DROPOUTS if (fw, d, s) not in done]
        have = len(SEEDS) * len(DROPOUTS) - len(todo)
        print(f"  {name:8s} 이미 {have}번 · 남은 {len(todo)}번 ≈ {len(todo) * MIN_PER_RUN[fw]:.0f}분")


plan()

## 셀 2 — Keras 5번 (Dropout 5개) · 약 13분

In [ ]:
import keras
from keras import layers


def make_optimizer(name, lr):
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "sgd_momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    return keras.optimizers.SGD(learning_rate=lr)


def show_epoch(epoch, logs):
    print(f"  에폭 {epoch + 1}/{EPOCHS}  "
          f"train {logs['accuracy']:.3f} (loss {logs['loss']:.3f})  ·  "
          f"val {logs['val_accuracy']:.3f} (loss {logs['val_loss']:.3f})", flush=True)


def run_keras(dropout, seed):
    keras.utils.set_random_seed(seed)

    def load(split, shuffle):
        ds = keras.utils.image_dataset_from_directory(
            f"{DATA_DIR}/{split}", image_size=(IMG, IMG),
            batch_size=BATCH, shuffle=shuffle, seed=seed)
        rescale = layers.Rescaling(1.0 / 255)
        return ds.map(lambda x, y: (rescale(x), y))

    train_ds = load("training", True)
    val_ds = load("validation", False)
    test_ds = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = keras.Sequential([
        keras.Input(shape=(IMG, IMG, 3)),

        layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dropout(dropout),
        layers.Dense(NUM_CLASSES, activation="softmax",
                     kernel_initializer="he_normal"),
    ])
    n_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_params == N_PARAMS, n_params

    model.compile(
        optimizer=make_optimizer(OPTIMIZER, LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    t0 = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0,
                  callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=show_epoch)]).history
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    hist = {"train_acc": h["accuracy"], "train_loss": h["loss"],
            "val_acc": h["val_accuracy"], "val_loss": h["val_loss"]}
    row = make_row("keras", dropout, seed, n_params, (time.time() - t0) / 60,
                   {k: [float(v) for v in vs] for k, vs in hist.items()},
                   float(test_loss), float(test_acc))
    keras.backend.clear_session()
    gc.collect()
    return row


for seed in SEEDS:
    for dropout in DROPOUTS:
        if ("keras", dropout, seed) in done_runs():
            print(f"{tag('keras', dropout, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('keras', dropout, seed)} 시작 (약 2.5분)", flush=True)
        row = run_keras(dropout, seed)
        save_row(row)
        report(row)

## 셀 3 — PyTorch 5번 (Dropout 5개) · 약 23분

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

tf_ = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),          # 0~1 변환 + HWC -> CHW
])


def make_optimizer(name, params, lr):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    return torch.optim.SGD(params, lr=lr)


def run_torch(dropout, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g = torch.Generator().manual_seed(seed)

    def load(split, shuffle):
        ds = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform=tf_)
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                          generator=g if shuffle else None,
                          num_workers=2, pin_memory=True)

    train_dl = load("training", True)
    val_dl = load("validation", False)
    test_dl = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(64 * 30 * 30, NUM_CLASSES),
    ).to(device)

    for m in model:       # 1단계 — He 초기화. 가중치만, bias는 PyTorch 기본값 그대로
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_params == N_PARAMS, n_params

    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_optimizer(OPTIMIZER, model.parameters(), LR)

    def evaluate(dataloader):
        model.eval()
        total, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                pred = model(X)
                total += loss_fn(pred, y).item() * y.size(0)
                correct += (pred.argmax(1) == y).sum().item()
                n += y.size(0)
        return total / n, correct / n

    hist = {"train_acc": [], "train_loss": [], "val_acc": [], "val_loss": []}
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        total, correct, n = 0.0, 0, 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total += loss.item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, val_acc = evaluate(val_dl)
        hist["train_acc"].append(correct / n)
        hist["train_loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        hist["val_loss"].append(val_loss)
        print(f"  에폭 {epoch + 1}/{EPOCHS}  "
              f"train {hist['train_acc'][-1]:.3f} (loss {hist['train_loss'][-1]:.3f})  ·  "
              f"val {val_acc:.3f} (loss {val_loss:.3f})", flush=True)

    test_loss, test_acc = evaluate(test_dl)
    row = make_row("torch", dropout, seed, n_params, (time.time() - t0) / 60,
                   hist, test_loss, test_acc)
    del model, optimizer, train_dl, val_dl, test_dl
    torch.cuda.empty_cache()
    gc.collect()
    return row


for seed in SEEDS:
    for dropout in DROPOUTS:
        if ("torch", dropout, seed) in done_runs():
            print(f"{tag('torch', dropout, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('torch', dropout, seed)} 시작 (약 4.5분)", flush=True)
        row = run_torch(dropout, seed)
        save_row(row)
        report(row)

## 셀 4 — 요약과 판정 (이 셀의 출력을 붙여 주면 된다)

In [ ]:
import unicodedata
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

with open(RESULTS, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
assert rows, "결과가 아직 없다 — 셀 2·3을 먼저 돌릴 것"
assert all(int(r["n_params"]) == N_PARAMS for r in rows), "파라미터 수가 다른 실행이 있다"

FW = {"keras": "Keras", "torch": "PyTorch"}
MIN_SEEDS = 3                  # 규칙을 적용하려면 조건마다 이만큼의 시드가 필요하다
REF = {                        # 2단계 Adam 1e-3 (Dropout 0.5), 5에폭 끝 값: (train acc, val acc, val loss 최저, test acc)
    "keras": dict(seed0=(77.6, 41.5, 1.71, 45.3), mean3=(75.4, 39.3, 1.74, 43.0)),
    "torch": dict(seed0=(66.8, 40.1, 1.77, 42.2), mean3=(67.2, 40.8, 1.78, 43.4)),
}


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def pad(s, w):
    return s + " " * max(0, w - width(s))


def bar(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def metrics(r):
    ta, tl = np.array(json.loads(r["train_acc"])), np.array(json.loads(r["train_loss"]))
    va, vl = np.array(json.loads(r["val_acc"])), np.array(json.loads(r["val_loss"]))
    ok = np.isfinite(vl)
    best = int(np.argmin(np.where(ok, vl, np.inf)))
    return dict(train_acc=ta[-1] * 100, val_acc=va[-1] * 100, gap=(ta[-1] - va[-1]) * 100,
                best_val_loss=float(vl[best]) if ok.any() else np.nan,
                best_ep=best + 1 if ok.any() else np.nan,
                test_acc=float(r["test_acc"]) * 100, test_loss=float(r["test_loss"]),
                diverged=bool(not np.isfinite(vl[-1]) or not np.isfinite(tl[-1])))


runs = {}      # (프레임워크, Dropout 비율) -> {시드: 지표}
curves = {}    # (프레임워크, Dropout 비율) -> {시드: 에폭별 곡선}
for r in rows:
    d = float(r["dropout"])
    if d not in DROPOUTS:      # 셀 1에서 뺀 비율의 예전 결과는 쓰지 않는다
        continue
    k, s = (r["framework"], d), int(r["seed"])
    runs.setdefault(k, {})[s] = metrics(r)
    curves.setdefault(k, {})[s] = dict(train_acc=np.array(json.loads(r["train_acc"])) * 100,
                                       val_loss=np.array(json.loads(r["val_loss"])))

fws = [fw for fw in FW if any(k[0] == fw for k in runs)]
assert fws, "Dropout 결과가 CSV에 없다 — 셀 1의 [계획]을 확인할 것"
levels = [d for d in DROPOUTS if any((fw, d) in runs for fw in fws)]
seeds_all = sorted({s for g in runs.values() for s in g})


def vals(group, key):
    v = np.array([m[key] for m in group.values()], dtype=float)
    return v[np.isfinite(v)]


def span(group, key, nd):
    v = vals(group, key)
    if len(v) == 0:
        return "-"
    if key == "best_ep" and len(group) == 1:
        nd = 0
    if len(group) == 1:
        return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f} [{v.min():.{nd}f}~{v.max():.{nd}f}]"


def mean_of(group, key):
    v = vals(group, key)
    return float(v.mean()) if len(v) else np.nan


def notes(d, group):
    out = []
    if d == BASE_DROPOUT:
        out.append("기준선")
    n_div = sum(m["diverged"] for m in group.values())
    if n_div:
        out.append(f"발산 {n_div}/{len(group)}")
    live = [m for m in group.values() if not m["diverged"]]
    n_last = sum(m["best_ep"] == EPOCHS for m in live)
    if live and n_last == len(live):
        out.append("최저 에폭=마지막")
    elif n_last:
        out.append(f"최저 에폭=마지막 {n_last}/{len(live)}")
    return " · ".join(out)


# ---- 결과표 -----------------------------------------------------------------
COLS = [("train_acc", "train acc %", 1), ("val_acc", "val acc %", 1), ("gap", "train − val %p", 1),
        ("best_val_loss", "val loss 최저", 2), ("best_ep", "최저 에폭", 1), ("test_acc", "test acc %", 1)]
for fw in fws:
    bar(f"[결과표] {FW[fw]} — Adam 1e-3, 5에폭 고정, 시드 {', '.join(map(str, seeds_all))}"
        + ("" if len(seeds_all) == 1 else ", 평균 [최소~최대]"))
    print("   val loss 최저 · 최저 에폭은 5에폭 중 val loss가 가장 낮았던 값이고, 나머지는 5에폭 끝 값이다")
    head = ["Dropout", "시드"] + [name for _, name, _ in COLS] + ["비고"]
    body = []
    for d in levels:
        g = runs.get((fw, d))
        if not g:
            body.append([str(d), "결과 없음"] + [""] * (len(head) - 2))
            continue
        body.append([str(d), ",".join(map(str, sorted(g)))]
                    + [span(g, key, nd) for key, _, nd in COLS] + [notes(d, g)])
    ws = [max(width(r[i]) for r in [head] + body) + 2 for i in range(len(head) - 1)]
    for r in [head] + body:
        print("".join(pad(c, w) for c, w in zip(r[:-1], ws)) + r[-1])
    r0, r3 = REF[fw]["seed0"], REF[fw]["mean3"]
    print("[참고] 2단계 Adam 1e-3 (Dropout 0.5) — "
          f"시드 0: train {r0[0]:.1f} · val {r0[1]:.1f} · val loss 최저 {r0[2]:.2f} · test {r0[3]:.1f}  /  "
          f"3시드 평균: train {r3[0]:.1f} · val {r3[1]:.1f} · val loss 최저 {r3[2]:.2f} · test {r3[3]:.1f}")


# ---- 같은 시드끼리 비교 -----------------------------------------------------
def paired(fw, a, b, key, higher_better):
    """같은 시드끼리 뺀 값 a-b로 비율 a와 b를 비교한다. higher_better=False면 값이 낮은 쪽이 낫다."""
    ga, gb = runs.get((fw, a), {}), runs.get((fw, b), {})
    seeds = [s for s in sorted(set(ga) & set(gb))
             if np.isfinite(ga[s][key]) and np.isfinite(gb[s][key])]
    out = dict(diffs=[], mean=np.nan, limit=np.nan, winner=None, enough=len(seeds) >= MIN_SEEDS,
               text="비교할 공통 시드가 없다")
    if not seeds:
        return out
    va = np.array([ga[s][key] for s in seeds])
    vb = np.array([gb[s][key] for s in seeds])
    d = va - vb
    out.update(diffs=[float(x) for x in d], mean=float(d.mean()))
    if not out["enough"]:
        out["text"] = f"판정 안 함 (공통 시드 {len(seeds)}개, {MIN_SEEDS}개 필요)"
        return out
    gain = d if higher_better else -d              # 양수 = a가 낫다
    limit = float(max(np.ptp(va), np.ptp(vb)))      # 시드 범위: 두 조건 중 큰 쪽
    out["limit"] = limit
    consistent = bool((gain > 0).all() or (gain < 0).all())
    if consistent and abs(float(gain.mean())) > limit:
        winner = "a" if gain.mean() > 0 else "b"
        out["winner"] = winner
        out["text"] = f"Dropout {a if winner == 'a' else b} 우세 (뚜렷함)"
    elif not consistent:
        out["text"] = (f"구분 안 됨 — 시드마다 방향이 다름 "
                       f"(Dropout {a} 우세 {int((gain > 0).sum())}/{len(seeds)})")
    else:
        out["text"] = "구분 안 됨 — 방향은 같지만 평균 차이가 시드 범위 안"
    return out


def show_cmp(fw, a, b, key, higher_better, title, nd, unit, judge=True):
    c = paired(fw, a, b, key, higher_better)
    head = pad(FW[fw], 9) + pad(f"Dropout {a}", 14) + pad(title, 18)
    if not c["diffs"]:
        print(head + c["text"])
        return c
    per = " ".join(f"{x:+.{nd}f}" for x in c["diffs"])
    lim = f"  시드 범위 {c['limit']:.{nd}f}" if c["enough"] else ""
    tail = f"  -> {c['text']}" if judge else "  (보고용)"
    print(head + f"{c['mean']:+.{nd}f}{unit}  (시드별 {per}){lim}" + tail)
    return c


bar(f"[기준선 대비] 같은 시드끼리 뺀 값 (Dropout 비율 − 기준선 {BASE_DROPOUT}), val loss는 낮은 쪽이 낫다\n"
    "   뚜렷함 = 시드 전부 같은 방향이고 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 것\n"
    "   시드가 3개 미만이면 판정하지 않고 차이만 보여 준다 (훑어보기)")
cells = {}
for d in levels:
    if d == BASE_DROPOUT:
        continue
    cells[d] = []
    for fw in fws:
        cells[d].append(show_cmp(fw, d, BASE_DROPOUT, "val_acc", True, "val acc", 1, "%p"))
        cells[d].append(show_cmp(fw, d, BASE_DROPOUT, "best_val_loss", False, "val loss 최저", 3, ""))
        show_cmp(fw, d, BASE_DROPOUT, "gap", False, "train − val 격차", 1, "%p", judge=False)
        show_cmp(fw, d, BASE_DROPOUT, "test_acc", True, "test acc", 1, "%p", judge=False)
    print()

if cells:
    print(f"-> 판정 (4칸 = val acc · val loss 최저 × 두 프레임워크, 기준선 {BASE_DROPOUT} 대비)")
movers = []
all_hold = True
for d, cs in cells.items():
    n_new = sum(c["winner"] == "a" for c in cs)
    n_base = sum(c["winner"] == "b" for c in cs)
    n_skip = sum(not c["enough"] for c in cs)
    n_tie = len(cs) - n_new - n_base - n_skip
    if len(fws) < 2 or n_skip:
        label = "판정 보류"
    else:
        all_hold = False
        if n_new == len(cs):
            label = "기준선보다 뚜렷하게 낫다 -> 기준선 후보"
            movers.append(d)
        elif n_base == len(cs):
            label = "기준선이 뚜렷하게 낫다"
        else:
            label = "구분 안 됨"
    extra = f" · 판정 안 함 {n_skip}칸" if n_skip else ""
    print(f"   Dropout {d}: {label}  (비율 우세 {n_new}칸 · 기준선 우세 {n_base}칸 · 구분 안 됨 {n_tie}칸{extra})")
print()
if not cells:
    print(f"-> 결론: 기준선(Dropout {BASE_DROPOUT})뿐이라 비교할 비율이 없다 — 위 표의 0.5 행이 [참고] 줄과 맞는지부터 본다")
elif all_hold:
    print("-> 결론: 판정 보류 — 두 프레임워크 모두 시드 3개씩이 필요하다. 지금은 훑어보기이므로 큰 차이만 읽는다")
elif movers:
    def score(d):
        return np.mean([mean_of(runs[(fw, d)], "val_acc") for fw in fws if (fw, d) in runs])
    pick = max(movers, key=score)
    print(f"-> 결론: 기준선을 Dropout {pick}로 옮긴다 (4칸 전부에서 기준선보다 뚜렷하게 앞섰다"
          + (f", 후보 {', '.join(str(m) for m in movers)} 중 val acc 평균이 가장 높다)" if len(movers) > 1 else ")"))
else:
    print(f"-> 결론: Dropout {BASE_DROPOUT} 유지 — 어느 비율도 4칸 전부에서 기준선보다 뚜렷하게 앞서지 않았다")

# ---- 그래프 -----------------------------------------------------------------
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
BLUE = "#2a78d6"
BLUES = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6",
         "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


def style(ax, grid_axis="y"):
    ax.set_facecolor(SURFACE)
    ax.grid(True, axis=grid_axis, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)


def ramp(n):
    if n == 1:
        return [BLUES[4]]
    return [BLUES[i] for i in np.linspace(0, len(BLUES) - 1, n).round().astype(int)]


# 그래프 1 — 비율에 따른 결과 (점 하나 = 시드 하나)
PANELS = [("val_acc", "val acc (%) at epoch 5", 1.0),
          ("best_val_loss", "best val loss (min over epochs)", 0.03),
          ("gap", "train - val acc gap (%p) at epoch 5", 1.0)]
fig, axes = plt.subplots(len(fws), 3, figsize=(13, 3.6 * len(fws) + 0.9), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (key, name, min_pad) in enumerate(PANELS):
        ax = axes[i][j]
        style(ax)
        seen = []
        for x, d in enumerate(levels):
            pts = [(s, m[key]) for s, m in sorted(runs.get((fw, d), {}).items()) if np.isfinite(m[key])]
            if not pts:
                continue
            ys = np.array([p[1] for p in pts])
            seen += list(ys)
            ax.vlines(x, ys.min(), ys.max(), color=BLUE, lw=1.6, alpha=0.45, zorder=2)
            ax.plot([x - 0.3, x + 0.3], [ys.mean()] * 2, color=BLUE, lw=2.4,
                    solid_capstyle="butt", zorder=3)
            ax.plot([x + 0.2 * (s - 1) for s, _ in pts], ys, ls="none", marker="o", ms=6.5,
                    mfc=BLUE, mec=SURFACE, mew=1.1, zorder=4)
        if seen:
            lo, hi = min(seen), max(seen)
            margin = max(min_pad, (hi - lo) * 0.2)
            ax.set_ylim(lo - margin, hi + margin)
        ax.set_xlim(-0.6, len(levels) - 0.4)
        ax.set_xticks(range(len(levels)))
        ax.set_xticklabels([str(d) + (" (base)" if d == BASE_DROPOUT else "") for d in levels])
        ax.set_xlabel("dropout rate", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=BLUE, lw=2.4, label="mean"),
           Line2D([0], [0], marker="o", ls="none", ms=6.5, mfc=BLUE, mec=SURFACE, mew=1.1,
                  label="each seed (left to right: 0, 1, 2)")]
fig.text(0.005, 0.99, "Dropout rate, Adam 1e-3, He init, 5 epochs",
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("dropout_seeds.png", facecolor=SURFACE)
plt.show()
print("[그래프] dropout_seeds.png")

# 그래프 2 — 에폭별 곡선 (선 하나 = Dropout 비율 하나, 연한 색 = 작은 비율)
CURVE_ROWS = [("train_acc", "train acc (%)"), ("val_loss", "val loss")]
fig, axes = plt.subplots(len(fws), 2, figsize=(10.5, 3.7 * len(fws) + 0.9), dpi=150,
                         squeeze=False, facecolor=SURFACE)
colors = ramp(len(levels))
for i, fw in enumerate(fws):
    for j, (key, name) in enumerate(CURVE_ROWS):
        ax = axes[i][j]
        style(ax, "both")
        for d, color in zip(levels, colors):
            c = curves.get((fw, d))
            if not c:
                continue
            arr = np.full((len(c), EPOCHS), np.nan)
            for n_, v in enumerate(c.values()):
                arr[n_, :len(v[key])] = v[key]
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                y = np.nanmean(arr, axis=0)
            ok = np.isfinite(y)
            ep = np.arange(1, EPOCHS + 1)
            ax.plot(ep[ok], y[ok], color=color, lw=2.6 if d == BASE_DROPOUT else 1.6, marker="o", ms=3.5,
                    mec=SURFACE, mew=0.8, solid_capstyle="round", zorder=3)
        ax.set_xlim(0.7, EPOCHS + 0.3)
        ax.set_xticks(range(1, EPOCHS + 1))
        ax.set_xlabel("epoch", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=c, lw=2.6 if d == BASE_DROPOUT else 1.6, marker="o", ms=3.5,
                  mec=SURFACE, mew=0.8, label=f"p={d}" + (" (base)" if d == BASE_DROPOUT else ""))
           for d, c in zip(levels, colors)]
fig.text(0.005, 0.99, "Curves by dropout rate (light = small, dark = large)  -  "
         + (f"seed {seeds_all[0]}" if len(seeds_all) == 1 else "mean over available seeds"),
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("dropout_curves.png", facecolor=SURFACE)
plt.show()
print("[그래프] dropout_curves.png")